# Ch.1 — Fragmentation monitoring (MM research memo)

**Kernel:** Python 3 + numpy, matplotlib. Run from `research/chapters/ch01_fragmentation/`.

## Executive takeaway

HL/Lit/RX ETH collector (~71 min): FEI_size≈**26%** (HL~91% size), FEI_updates≈**82%**, consolidated book **crossed ~100%** (mean cons. spread ≈−6.2 bps), HL **~99% 1-tick**.

| Decision | Items |
|----------|-------|
| **Promote** | `frag.crossed_nbbo` (E-gate), `tick.*`, `frag.update_share` |
| **Hold** | `frag.fei_tob_size`, spatial `frag.fei_trade`, SOR split |
| **Kill** | `frag.duplicate_best`, dark toxicity |


## Theory & formulas

$$\mathrm{FEI}=H(q)/\log N,\quad q_v\propto b_v+a_v\ \mathrm{(TOB\ coin)},\quad s=10^4(A-B)/M\ \mathrm{(bps)}$$

Crossed: $B^*=\max B$, $A^*=\min A$, crossed if $B^*>A^*$. **Honesty:** TOB FEI ≠ trade market share; crossed ≠ arb PnL.


## Data & method

Collector `results/xarb_md/tob/20260929/`, 1s alignment. Outputs in `research/out/ch01_fragmentation/`.


In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
HERE = Path.cwd().resolve()
OUT = next(p for p in [HERE/"../../out/ch01_fragmentation", HERE/"out/ch01_fragmentation", HERE/"research/out/ch01_fragmentation"] if p.exists()).resolve()
eth = json.loads((OUT/"exp_ch01_eth_summary.json").read_text())
btc = json.loads((OUT/"exp_ch01_btc_summary.json").read_text())
s, ex = eth["summary"], eth["extras"]
venues = list(s["size_share"].keys())
print("FEI_size=%.3f crossed=%.3f" % (s["fei_size"], ex["crossed_book_frac"]))


In [ ]:
size = [s["size_share"][v] for v in venues]
upd = [s["update_share"][v] for v in venues]
x = np.arange(len(venues)); w = 0.35
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - w/2, size, w, label="TOB size share")
ax.bar(x + w/2, upd, w, label="update share")
ax.set_xticks(x); ax.set_xticklabels(venues)
ax.set_ylabel("Share"); ax.set_ylim(0, 1)
ax.set_title("ETH FEI(size)=%.1f%% vs FEI(upd)=%.1f%%" % (100*s["fei_size"], 100*s["fei_updates"]))
ax.legend(frameon=False); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(venues, [s["mean_spread_bps"][v] for v in venues])
axes[0].set_ylabel("bps"); axes[0].set_title("Mean spread")
axes[1].bar(venues, [s["frac_one_tick"][v] for v in venues])
axes[1].set_ylim(0, 1); axes[1].set_title("P(1-tick)")
plt.tight_layout(); plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
xx = np.arange(len(venues))
axes[0].bar(xx - 0.2, [s["nbbo_bid_frac"][v] for v in venues], 0.4, label="NBBO bid")
axes[0].bar(xx + 0.2, [s["nbbo_ask_frac"][v] for v in venues], 0.4, label="NBBO ask")
axes[0].set_xticks(xx); axes[0].set_xticklabels(venues)
axes[0].legend(frameon=False); axes[0].set_title("NBBO participation")
risk = {"dup bid": s["duplicate_bid_frac"], "dup ask": s["duplicate_ask_frac"], "crossed": ex["crossed_book_frac"]}
axes[1].bar(list(risk.keys()), list(risk.values()), color=["#888888", "#888888", "#aa3333"])
axes[1].set_ylim(0, 1.05)
axes[1].set_title("SOR risk · cons. spread %.2f bps" % ex["consolidated_spread_bps_mean"])
plt.tight_layout(); plt.show()


## MM interpretation

Promote crossed as **gate**, ticks as **quote regime**, update-share as **flicker monitor**. Hold TOB FEI for inventory sizing. Kill duplicate-best on this panel.

See `EXP_REPORT.md`, `CANDIDATES.md`.
